In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.primitives import StatevectorSampler
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
import time
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer import AerSimulator
from qiskit.quantum_info import partial_trace
from iqm.qiskit_iqm.move_gate import MOVE_GATE_UNITARY
from sklearn.datasets import load_digits
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

SHOTS = 8192

# Noiseless simulator that returns shots
ideal_sampler = StatevectorSampler(seed=42)

# Aer device for the noisy simulations: GPU if available (needs the qiskit-aer-gpu package), otherwise CPU
USE_GPU = True
AER_DEVICES = AerSimulator().available_devices()
DEVICE = "GPU" if USE_GPU and "GPU" in AER_DEVICES else "CPU"
print("Aer devices:", AER_DEVICES, "-> using", DEVICE)

1. VLQ calibration data

VLQ has a **star topology**: 24 qubits (QB1 ... QB24) connected to a central resonator (COMPR1), not to each other. A two-qubit gate between qubits a and b is done as MOVE (a -> resonator), CZ (b, resonator), MOVE (resonator -> a).

data from file `vlq-calset-default-metrics.json`

In [ ]:
# Placeholder values
N_QB = 24

PRX_TIME_NS  = 40                         # single-qubit gate (r) duration
CZ_TIME_NS   = 60                         # CZ (qubit-resonator) duration
MOVE_TIME_NS = 100                        # MOVE (qubit <-> resonator) duration

T1_us     = np.full(N_QB, 40.0)           # energy relaxation time T1 (µs)
T2_us     = np.full(N_QB, 30.0)           # dephasing time T2 (µs)
err_1q    = np.full(N_QB, 1e-3)           # single-qubit gate error (1 - fidelity)
p1_given0 = np.full(N_QB, 0.02)           # readout error P(1|0)
p0_given1 = np.full(N_QB, 0.04)           # readout error P(0|1)
err_cz    = np.full(N_QB, 1e-2)           # CZ error between QB(i+1) and the resonator
err_move  = np.full(N_QB, 1e-2)           # MOVE error between QB(i+1) and the resonator

T1_res_us, T2_res_us = 40.0, 30.0         # resonator coherence times (µs)

for arr in [T1_us, T2_us, err_1q, p1_given0, p0_given1, err_cz, err_move]:
    assert len(arr) == N_QB

1b. Read the calibration file

File: `GET /api/v1/calibration-sets/star24/default/metrics` from https://extranet.it4i.cz/vlq-calibration/

Metrics used (QBx = qubit, COMPR1 = resonator):
- T1, T2: `characterization.model.QBx.t1_time`, `t2_time` (in seconds; if T2 is missing, `t2_echo_time` is used)
- single-qubit gate: `metrics.rb.prx.drag_crf.QBx.fidelity:par=d1` (randomized benchmarking)
- readout: `metrics.ssro.measure.constant.QBx.error_0_to_1`, `error_1_to_0`
- CZ: `metrics.irb.cz.crf_crf.QBx__COMPR1.fidelity` (interleaved RB)
- MOVE: `metrics.irb.move.crf_crf.QBx__COMPR1.fidelity:n_interleaved_gates=2`; the interleaved block is 2 MOVEs (there and back), so the error of one MOVE is (1 - F) / 2
- resonator: `characterization.model.COMPR1.t1_time`, `t2_time`

In [ ]:
import json

with open("vlq-calset-default-metrics.json") as f:
    calibration = json.load(f)

metrics = {obs["dut_field"]: obs["value"] for obs in calibration["observations"] if not obs["invalid"]}
print("Calibration date:", calibration["created_timestamp"])

for i in range(N_QB):
    q = f"QB{i + 1}"
    T1_us[i] = metrics[f"characterization.model.{q}.t1_time"] * 1e6
    T2_us[i] = metrics.get(f"characterization.model.{q}.t2_time",
                           metrics[f"characterization.model.{q}.t2_echo_time"]) * 1e6
    err_1q[i]    = 1 - metrics[f"metrics.rb.prx.drag_crf.{q}.fidelity:par=d1"]
    p1_given0[i] = metrics[f"metrics.ssro.measure.constant.{q}.error_0_to_1"]
    p0_given1[i] = metrics[f"metrics.ssro.measure.constant.{q}.error_1_to_0"]
    err_cz[i]    = 1 - metrics[f"metrics.irb.cz.crf_crf.{q}__COMPR1.fidelity"]
    err_move[i]  = (1 - metrics[f"metrics.irb.move.crf_crf.{q}__COMPR1.fidelity:n_interleaved_gates=2"]) / 2

T1_res_us = metrics["characterization.model.COMPR1.t1_time"] * 1e6
T2_res_us = metrics["characterization.model.COMPR1.t2_time"] * 1e6

print(f"{'qubit':6s} {'T1(us)':>7s} {'T2(us)':>7s} {'err_1q':>8s} {'P(1|0)':>7s} {'P(0|1)':>7s} {'err_cz':>7s} {'err_move':>8s}")
for i in range(N_QB):
    print(f"QB{i + 1:<4d} {T1_us[i]:7.1f} {T2_us[i]:7.1f} {err_1q[i]:8.4f} {p1_given0[i]:7.3f} "
          f"{p0_given1[i]:7.3f} {err_cz[i]:7.4f} {err_move[i]:8.4f}")
print(f"COMPR1 {T1_res_us:7.1f} {T2_res_us:7.1f}")

# Qubits used in the simulation: ranked by two-qubit (CZ + 2 MOVE) and readout errors
score = err_cz + 2 * err_move + (p1_given0 + p0_given1) / 2
BEST_QUBITS = [f"QB{i + 1}" for i in np.argsort(score)]
print("\nQubits ranked from best to worst:", BEST_QUBITS)

2. Star-topology circuits and noise model
Noise model needs to know which physical qubit each gate acts on. `transpile_to_IQM` renumbers qubits unreliably → can't assign errors per gate. Transpile by hand. Noise after each gate, with that qubit's calibration:
  - `thermal_relaxation_error` (T1, T2)
  - `depolarizing_error` (gate fidelity)
  - `ReadoutError`
- `t_scale`: multiplies T1, T2 (coherence sweep)

In [ ]:
def to_star(qc_cz, n):
    # qc_cz: circuit on n data qubits with native gates `rz`, `sx`, `x`, `cz` (CZs between data qubits)
    # returns the same circuit on n data qubits + resonator (index n), with CZs routed through the resonator
    R = n
    star = QuantumCircuit(n + 1)
    if qc_cz.num_clbits: 
        star.add_register(ClassicalRegister(qc_cz.num_clbits, "meas"))
    holder = None                                    # data qubit whose state is in the resonator

    def move_out():
        nonlocal holder
        if holder is not None:
            star.unitary(MOVE_GATE_UNITARY, [holder, R], label="move")
            holder = None

    for inst in qc_cz.data:
        name = inst.operation.name
        qubits = [qc_cz.find_bit(q).index for q in inst.qubits]
        if name == "barrier":
            continue
        if name == "cz":
            a, b = qubits
            if holder == b:
                a, b = b, a
            if holder != a:                          # bring a into the resonator
                move_out()
                star.unitary(MOVE_GATE_UNITARY, [a, R], label="move")
                holder = a
            star.cz(b, R)                            # CZ(b, resonator) = CZ(b, a)
        elif name == "measure":
            move_out()
            star.measure(qubits[0], qc_cz.find_bit(inst.clbits[0]).index)
        else:                                        # single-qubit gate
            if holder in qubits:
                move_out()
            star.append(inst.operation, qubits)
    move_out()
    return star


def calibration_index(name):
    # component name -> index in the calibration arrays (None for the resonator)
    return int(name[2:]) - 1 if name.startswith("QB") else None


def build_noise_model(names, t_scale=1.0, gate_errors=True, readout_errors=True):
    # names[k] = component (QBx or COMPR1) of qubit k of the simulated circuit
    noise_model = NoiseModel(basis_gates=["rz", "sx", "x", "cz", "unitary"])

    def relaxation(p, time_ns):
        c = calibration_index(p)
        T1 = (T1_us[c] if c is not None else T1_res_us) * 1e3 * t_scale   # µs -> ns
        T2 = (T2_us[c] if c is not None else T2_res_us) * 1e3 * t_scale
        return thermal_relaxation_error(T1, min(T2, 2 * T1), time_ns)

    for k, p in enumerate(names):
        c = calibration_index(p)
        if c is None:                                   # resonator: no single-qubit gates, no readout
            continue

        # single-qubit gates sx, x (rz is virtual: no error)
        error = relaxation(p, PRX_TIME_NS)
        if gate_errors:
            error = depolarizing_error(2 * err_1q[c], 1).compose(error)
        noise_model.add_quantum_error(error, ["sx", "x"], [k])

        # readout
        if readout_errors:
            e0, e1 = p1_given0[c], p0_given1[c]
            noise_model.add_readout_error(ReadoutError([[1 - e0, e0], [e1, 1 - e1]]), [k])

    # two-qubit gates: always between a qubit and the resonator
    for k0, p0 in enumerate(names):
        for k1, p1 in enumerate(names):
            if k0 == k1:
                continue
            c = calibration_index(p0) if calibration_index(p0) is not None else calibration_index(p1)
            for gate, time_ns, err in [("cz", CZ_TIME_NS, err_cz), ("move", MOVE_TIME_NS, err_move)]:
                error = relaxation(p0, time_ns).expand(relaxation(p1, time_ns))
                if gate_errors:
                    error = depolarizing_error(4 / 3 * err[c], 2).compose(error)
                noise_model.add_quantum_error(error, gate, [k0, k1])

    return noise_model


VLQ_NOISE = dict(t_scale=1.0, gate_errors=True, readout_errors=True)

3. Dataset (0,2,7,9)

Binary task: two digits chosen at random among the four. 
Multiclass task: the four digits.

In [ ]:
DATASET = "mnist"                               
N_TRAIN, N_TEST = 1000, 1000                       # images per digit

from sklearn.datasets import fetch_openml
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
data, target = mnist.data / 255.0, mnist.target.astype(int)


PAN_DIGITS = [0, 2, 7, 9]

rng_pair = np.random.default_rng(seed=0)
binary_digits = sorted(rng_pair.choice(PAN_DIGITS, size=2, replace=False).tolist())

tasks = {f"{binary_digits[0]} vs {binary_digits[1]}": binary_digits,
         "0, 2, 7, 9": PAN_DIGITS}
print(tasks)


def make_split(digit_list, seed=0):
    rng_digits = np.random.default_rng(seed)
    train_idx, test_idx = [], []
    for d in digit_list:
        idx_d = np.where(target == d)[0]              # all images of digit d
        assert len(idx_d) >= N_TRAIN + N_TEST, f"only {len(idx_d)} images of digit {d}"
        idx_d = rng_digits.permutation(idx_d)         # shuffle them
        train_idx += list(idx_d[:N_TRAIN])
        test_idx  += list(idx_d[N_TRAIN:N_TRAIN + N_TEST])
    return data[train_idx], target[train_idx], data[test_idx], target[test_idx]

4. Compression to n features: truncated SVD + scaling to [0, 1]

In [ ]:
def compression(X_train_raw, X_test_raw, n):
    svd = TruncatedSVD(n_components=n, random_state=0)
    svd.fit(X_train_raw)                         # learns the n directions, using only train
    Z_train = svd.transform(X_train_raw)
    Z_test  = svd.transform(X_test_raw)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(Z_train)                          # learns min and max, using only train
    X_train = scaler.transform(Z_train)
    X_test  = np.clip(scaler.transform(Z_test), 0, 1)
    return X_train, X_test

5. Kernel circuit: U(x_j) followed by U(x_i)^dagger

In [ ]:
def feature_map(x, n):          # n = number of qubits = number of features
    qc = QuantumCircuit(n)

    # Feature map, 4 layers 
    for _ in range(4):
        for i in range(n):
            qc.h(i)
            qc.rz(2 * x[i], i)
        for i in range(n - 1):          # RZZ on neighbouring pairs (0,1), (1,2), ...
            x_ij = 2 * (np.pi - x[i]) * (np.pi - x[i + 1])
            qc.rzz(x_ij, i, i + 1)
    return qc


def kernel_circuit(x_i, x_j, n):
    qc = feature_map(x_j, n)                                 # U(x_j)
    qc.compose(feature_map(x_i, n).inverse(), inplace=True)  # U(x_i)^dagger
    qc.measure_all()
    return qc

6. Kernel from the quantum states (one simulation per image)

The kernel is computed from the overlaps of the states:

- **Noiseless**: |φ(x)> = U(x)|0>, K(x, x') = |<φ(x)|φ(x')>|².
- **Noisy**: ρ(x) = state prepared by U(x) **on the VLQ noise model** , K(x, x') = Tr[ρ(x) ρ(x')]. 

The shot statistics (8192 shots) is reproduced by drawing each kernel entry from a binomial distribution.

In [ ]:
# Quick check: statevector kernel vs the compute-uncompute circuit on the sampler
x_a, x_b = np.array([0.2, 0.7, 0.4]), np.array([0.5, 0.1, 0.9])
S = statevectors([x_a, x_b], 3)
print("statevector :", abs(S[0].conj() @ S[1]) ** 2)
r = ideal_sampler.run([kernel_circuit(x_a, x_b, 3)], shots=100000).result()[0]
print("sampler     :", r.data.meas.get_counts().get("000", 0) / 100000)

# noisy states: trace must be 1 and purity < 1
R = noisy_states([x_a, x_b], 3, VLQ_NOISE)
print("trace       :", np.real(np.trace(R[0].reshape(8, 8))), " purity:", np.real(R[0] @ R[0].conj()))

Circuit used (n = 3)

In [ ]:
n_show = 3
feature_map(ParameterVector("x", n_show), n_show).draw("mpl")

In [ ]:
qc_dm, names, _, _ = noisy_state_template(n_show)
print("components:", names)
print("gate counts:", dict(qc_dm.count_ops()))
qc_dm.draw("mpl", fold=40)

7. General case: n = 2, ..., 9

In [ ]:
def case2_gen(n, digit_list, noise):
    X_train_raw, y_train, X_test_raw, y_test = make_split(digit_list)
    X_train, X_test = compression(X_train_raw, X_test_raw, n)

    K_train, K_test = kernel_matrices(X_train, X_test, n, noise)

    svm = SVC(kernel="precomputed")
    svm.fit(K_train, y_train)
    acc = svm.score(K_test, y_test)
    return acc, K_train


def nmse(K_sim, K):                              # Eq. (3) of Melo et al.
    return np.sum((K_sim - K) ** 2) / np.sum(K_sim ** 2)

8. Noiseless vs noisy simulation (Fig. 4(a), 4(b), 4(d) of Melo et al.)

Results are saved after every n. The name includes the number of images.

In [ ]:
n_list = [2, 3, 4, 5, 6, 7, 8, 9]    # number of qubits

results = {}
for task, digit_list in tasks.items():
    results[task] = {"acc_ideal": [], "acc_noisy": [], "nmse": [], "K_ideal": {}, "K_noisy": {}}
    for n in n_list:
        t0 = time.time()
        acc_ideal, K_ideal = case2_gen(n, digit_list, noise=None)
        acc_noisy, K_noisy = case2_gen(n, digit_list, noise=VLQ_NOISE)

        res = results[task]
        res["acc_ideal"].append(acc_ideal)
        res["acc_noisy"].append(acc_noisy)
        res["nmse"].append(nmse(K_ideal, K_noisy))
        res["K_ideal"][n] = K_ideal
        res["K_noisy"][n] = K_noisy
        print(f"{task} | n = {n}: noiseless = {acc_ideal:.2f}, noisy = {acc_noisy:.2f}, "
              f"NMSE = {res['nmse'][-1]:.3f} ({(time.time() - t0) / 60:.1f} min)")

        np.save(f"results_case2_noise_{DATASET}_{N_TRAIN}_{N_TEST}.npy", {"n_list": n_list, "tasks": tasks, "results": results},
                allow_pickle=True)

In [ ]:
# Fig. 4(a) and 4(b): accuracy and NMSE vs number of qubits
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for k, (task, res) in enumerate(results.items()):
    ax[k].plot(n_list, res["acc_ideal"], "o-", label="Noiseless simulation")
    ax[k].plot(n_list, res["acc_noisy"], "o-", label="VLQ noise model")
    ax[k].set_title(task)
    ax[k].set_ylabel("Classification accuracy")
    ax[2].plot(n_list, res["nmse"], "o-", label=task)
ax[2].set_ylabel("Normalized MSE")
for a in ax:
    a.set_xlabel("Number of qubits"); a.set_xticks(n_list); a.legend()
plt.tight_layout()
plt.savefig("fig_noise_accuracy.jpg", bbox_inches="tight")
plt.show()

# Fig. 4(d) and 4(f): training kernel matrices, n = 9, four digits
task = "0, 2, 7, 9"
fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
for a, key, title in zip(ax, ["K_ideal", "K_noisy"], ["Noiseless simulation", "VLQ noise model"]):
    im = a.imshow(results[task][key][9], cmap="gray_r", vmin=0, vmax=1)
    a.set_title(f"{title}, n = 9")
    plt.colorbar(im, ax=a)
plt.tight_layout()
plt.savefig("fig_noise_kernels.jpg", bbox_inches="tight")
plt.show()